In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("D:\PowerBI Projects\Pharma_Dashboard\Data\cleaned_pharma_sales.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_27692\2157900171.py:1: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv("D:\PowerBI Projects\Pharma_Dashboard\Data\cleaned_pharma_sales.csv")


,Distributor,Customer Name,City,Country,Latitude,Longitude,Channel,Sub-channel,Product Name,Product Class,Quantity,Price,Sales,Month,Year,Name of Sales Rep,Manager,Sales Team
0,Gottlieb-Cruickshank,"Zieme, Doyle and Kunze",Lublin,Poland,51.2333,22.5667,Hospital,Private,Topipizole,Mood Stabilizers,4.0,368,1472.0,January,2018,Mary Gerrard,Britanny Bold,Delta
1,Gottlieb-Cruickshank,Feest PLC,Świecie,Poland,53.4167,18.4333,Pharmacy,Retail,Choriotrisin,Antibiotics,7.0,591,4137.0,January,2018,Jessica Smith,Britanny Bold,Delta
2,Gottlieb-Cruickshank,Medhurst-Beer Pharmaceutical Limited,Rybnik,Poland,50.0833,18.5000,Pharmacy,Institution,Acantaine,Antibiotics,30.0,66,1980.0,January,2018,Steve Pepple,Tracy Banks,Bravo
3,Gottlieb-Cruickshank,Barton Ltd Pharma Plc,Czeladź,Poland,50.3333,19.0833,Hospital,Private,Lioletine Refliruvax,Analgesics,6.0,435,2610.0,January,2018,Mary Gerrard,Britanny Bold,Delta
4,Gottlieb-Cruickshank,Keeling LLC Pharmacy,Olsztyn,Poland,53.7800,20.4942,Pharmacy,Retail,Oxymotroban Fexoformin,Analgesics,20.0,458,9160.0,January,2018,Anne Wu,Britanny Bold,Delta


In [7]:
customer = (
    df.groupby("Customer Name")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Quantity=("Quantity", "sum"),
          Product_Diversity=("Product Name", "nunique"),
          Average_Price=("Price", "mean")
      )
      .reset_index()
)

customer.head()

,Customer Name,Total_Sales,Total_Quantity,Product_Diversity,Average_Price
0,Abernathy Group,24876667.0,60005.0,184,414.291785
1,Abernathy Group Pharmaceutical Limited,2079222.0,5490.0,151,398.681614
2,Abernathy Group Pharmacy,34635271.0,83506.0,188,419.338462
3,Abernathy and Sons,20722227.0,50528.0,192,401.489362
4,Abernathy and Sons Pharmaceutical Limited,18857078.0,46832.0,194,403.718670


In [8]:
customer["Sales_Score"] = pd.qcut(
    customer["Total_Sales"],
    q=4,
    labels=[1,2,3,4]
).astype(int)

customer["Quantity_Score"] = pd.qcut(
    customer["Total_Quantity"],
    q=4,
    labels=[1,2,3,4]
).astype(int)

customer["Product_Score"] = pd.qcut(
    customer["Product_Diversity"],
    q=4,
    labels=[1,2,3,4]
).astype(int)

In [9]:
customer["Segment_Score"] = (
      customer["Sales_Score"] * 0.5
    + customer["Quantity_Score"] * 0.3
    + customer["Product_Score"] * 0.2
)

In [10]:
customer["Customer_Segment"] = pd.qcut(
    customer["Segment_Score"],
    q=4,
    labels=[
        "Bronze",
        "Silver",
        "Gold",
        "Platinum"
    ]
)

In [11]:
customer["Customer_Segment"].value_counts()

Customer_Segment
Gold        192
Bronze      191
Silver      185
Platinum    183
Name: count, dtype: int64

In [12]:
segment_summary = (
    customer.groupby("Customer_Segment")
    .agg(
        Customers=("Customer Name","count"),
        Revenue=("Total_Sales","sum"),
        Avg_Revenue=("Total_Sales","mean"),
        Avg_Quantity=("Total_Quantity","mean"),
        Avg_Product_Diversity=("Product_Diversity","mean")
    )
)

segment_summary

C:\Users\ASUS\AppData\Local\Temp\ipykernel_27692\3737414588.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  customer.groupby("Customer_Segment")


,Customers,Revenue,Avg_Revenue,Avg_Quantity,Avg_Product_Diversity
Customer_Segment,,,,,
Bronze,191,6.238930e+08,3.266455e+06,8091.953927,136.157068
Silver,185,2.562402e+09,1.385082e+07,35050.542777,188.156757
Gold,192,3.569161e+09,1.858938e+07,45958.000205,192.083333
Platinum,183,5.043522e+09,2.756023e+07,64616.771309,193.737705


In [13]:
customer.sort_values(
    "Total_Sales",
    ascending=False
).head(20)

,Customer Name,Total_Sales,Total_Quantity,Product_Diversity,Average_Price,Sales_Score,Quantity_Score,Product_Score,Segment_Score,Customer_Segment
460,Mraz-Kutch Pharma Plc,9.356178e+07,174234.000000,189,408.835979,4,4,2,3.6,Platinum
497,"Parker, Green and Emmerich Pharma Plc",6.489050e+07,132080.000000,201,422.932161,4,4,4,4.0,Platinum
745,Zemlak-Witting,5.600668e+07,124868.000000,188,408.163212,4,4,2,3.6,Platinum
449,"Moen, Murazik and Smith Pharm",5.088532e+07,94970.000000,195,410.787342,4,4,3,3.8,Platinum
446,Mills Inc Pharmaceutical Ltd,4.593901e+07,89970.000000,187,424.825974,4,4,2,3.6,Platinum
654,Streich PLC,4.445462e+07,103978.000000,196,413.947514,4,4,4,4.0,Platinum
214,"Gleichner, Bahringer and Morar Pharmaceutical ...",4.377938e+07,100146.000000,188,411.371134,4,4,2,3.6,Platinum
534,Reichel Inc,4.349563e+07,90563.000000,194,411.787709,4,4,3,3.8,Platinum
562,"Romaguera, Moen and Hagenes Pharmacy",4.206666e+07,91253.734824,191,419.826190,4,4,3,3.8,Platinum
663,"Torphy, Pfeffer and Jakubowski Pharm",4.151422e+07,85885.000000,183,399.184416,4,4,2,3.6,Platinum


In [14]:
customer.to_csv(
    "../Data/customer_segmentation.csv",
    index=False
)

print("Customer segmentation exported successfully.")

Customer segmentation exported successfully.


In [15]:
print("""
Business Insights
-----------------

• Platinum customers generate the highest revenue and purchase the
  widest range of pharmaceutical products.

• Gold customers represent strong growth opportunities and should
  receive targeted commercial engagement.

• Bronze customers contribute relatively less revenue and may benefit
  from focused promotional strategies.

• Customer segmentation enables sales teams to prioritize accounts,
  optimize resource allocation, and improve commercial planning.
""")


Business Insights
-----------------

• Platinum customers generate the highest revenue and purchase the
  widest range of pharmaceutical products.

• Gold customers represent strong growth opportunities and should
  receive targeted commercial engagement.

• Bronze customers contribute relatively less revenue and may benefit
  from focused promotional strategies.

• Customer segmentation enables sales teams to prioritize accounts,
  optimize resource allocation, and improve commercial planning.



In [19]:
final_df = df.merge(
    customer[["Customer Name", "Customer_Segment"]],
    on="Customer Name",
    how="left"
)

final_df.to_csv(
    "../Data/pharma_segmented_new.csv",
    index=False
)

print("Power BI dataset with customer segments created successfully.")

Power BI dataset with customer segments created successfully.


In [17]:
import pandas as pd

df = pd.read_csv("../Data/pharma_segmented.csv")

print(df.columns.tolist())

print(len(df.columns))

['Distributor', 'Customer Name', 'City', 'Country', 'Latitude', 'Longitude', 'Channel', 'Sub-channel', 'Product Name', 'Product Class', 'Quantity', 'Price', 'Sales', 'Month', 'Year', 'Name of Sales Rep', 'Manager', 'Sales Team', 'Customer_Segment']
19


In [18]:
print(customer.columns)
print(final_df.columns)

Index(['Customer Name', 'Total_Sales', 'Total_Quantity', 'Product_Diversity',
       'Average_Price', 'Sales_Score', 'Quantity_Score', 'Product_Score',
       'Segment_Score', 'Customer_Segment'],
      dtype='object')
Index(['Distributor', 'Customer Name', 'City', 'Country', 'Latitude',
       'Longitude', 'Channel', 'Sub-channel', 'Product Name', 'Product Class',
       'Quantity', 'Price', 'Sales', 'Month', 'Year', 'Name of Sales Rep',
       'Manager', 'Sales Team', 'Customer_Segment'],
      dtype='object')
